## Import NumPy

In [1]:
import numpy as np

## Load Experimental Data

Load the file generated by the Rust code.

In [2]:
data = np.loadtxt(
    "./x_ancilla_probs.csv",
    delimiter=",",
)

The `bitstrings` correspond to the 4 $X-$ ancilla checks $[a_0,a_1,a_2,a_3]$. `probs` contains the probability of measuring a given ancilla reading.

In [3]:
bitstrings = data[:, 0:4].astype(int)
probs = data[:, 4]

## Calculating Detector Counts

Since we projected into the $\ket{+}_L$ subspace, we only need to now tally the probabilities to obtain $\braket{v_i}$ , $\braket{v_iv_j}$ and $\braket{v_i \otimes v_j}$ 

In [4]:
N_AUX_QUBITS = 4
count_arr = np.zeros((N_AUX_QUBITS, N_AUX_QUBITS), dtype=float)
count_xor = np.zeros((N_AUX_QUBITS, N_AUX_QUBITS), dtype=float)

for p in range(len(probs)):
    bitstring = bitstrings[p]
    prob = probs[p]
    for i in range(N_AUX_QUBITS):
        for j in range(N_AUX_QUBITS):
            if bitstring[i] == 1 and bitstring[j] == 1:
                count_arr[i][j] += prob
            if (bitstring[i] ^ bitstring[j]) == 1:
                count_xor[i][j] += prob

We now calculate $p_{ij}$ and store it in `p_arr`

In [5]:
p_arr = np.zeros((N_AUX_QUBITS, N_AUX_QUBITS), dtype=float)

for i in range(N_AUX_QUBITS):
    for j in range(N_AUX_QUBITS):
        if i != j and abs(i - j) == 1:
            p_arr[i, j] = 0.5 - np.sqrt(
                0.25
                - (count_arr[i, j] - count_arr[i, i] * count_arr[j, j])
                / (1.0 - 2.0 * count_xor[i, j])
            )

for i in range(N_AUX_QUBITS):
    denom = 1.0
    for j in range(N_AUX_QUBITS):
        if i != j and abs(i - j) == 1:
            denom *= 1.0 - 2.0 * p_arr[i, j]
    p_arr[i, i] = 0.5 + (count_arr[i, i] - 0.5) / denom

Calculate inferred $\tilde{\theta}_{ij}$ from $p_{ij}=\sin^2 \tilde{\theta}_{ij}$

In [6]:
def calculate_angle(p):
    return (1.0 / np.pi) * np.arcsin(np.sqrt(p))


angle_arr = np.vectorize(calculate_angle)(p_arr)

In [7]:
from IPython.display import display, Markdown

for i in range(N_AUX_QUBITS):
    for j in range(i, N_AUX_QUBITS):
        if angle_arr[i, j] != 0.0:
            display(
                Markdown(
                    f"edge {i} $\\leftrightarrow$ {j}:\t" + r"$\tilde{\theta}$="
                    f"\t{angle_arr[i, j]:.6f} $\\pi$ \n"
                )
            )

edge 0 $\leftrightarrow$ 0:	$\tilde{\theta}$=	0.100000 $\pi$ 


edge 0 $\leftrightarrow$ 1:	$\tilde{\theta}$=	0.100000 $\pi$ 


edge 1 $\leftrightarrow$ 1:	$\tilde{\theta}$=	0.200000 $\pi$ 


edge 1 $\leftrightarrow$ 2:	$\tilde{\theta}$=	0.100000 $\pi$ 


edge 2 $\leftrightarrow$ 2:	$\tilde{\theta}$=	0.200000 $\pi$ 


edge 2 $\leftrightarrow$ 3:	$\tilde{\theta}$=	0.100000 $\pi$ 


edge 3 $\leftrightarrow$ 3:	$\tilde{\theta}$=	0.100000 $\pi$ 
